In [ ]:
# ขั้นตอนที่ 1: ติดตั้งไลบรารี kaggle (ปกติ Colab มีให้อยู่แล้ว แต่รันกันเหนียวไว้ก่อน)
!pip install -q kaggle

# ขั้นตอนที่ 2: สร้างกล่องให้อัปโหลดไฟล์ ให้เราเลือกไฟล์ kaggle.json ที่เพิ่งโหลดมาจากหน้าเว็บ
from google.colab import files
uploaded = files.upload()



KeyboardInterrupt: 

In [ ]:
# ขั้นตอนที่ 3: จัดการไฟล์ยืนยันตัวตน (Authentication Setup)
# - สร้างโฟลเดอร์ชื่อ .kaggle
# - ก๊อปปี้ไฟล์ json เข้าไป
# - เปลี่ยนสิทธิ์ (chmod 600) ให้เจ้าของไฟล์อ่าน/เขียนได้คนเดียว เพื่อความปลอดภัย
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ขั้นตอนที่ 4: สั่งดาวน์โหลดชุดข้อมูลของการแข่งขัน Word Segmentation
# ใส่ parameter -c ตามด้วยชื่อการแข่งขัน
!kaggle competitions download -c super-ai-engineer-ss-6-word-segmentation

# ขั้นตอนที่ 5: แตกไฟล์ ZIP ที่ได้มา เอาไปไว้ในโฟลเดอร์ชื่อ 'dataset' เพื่อความเป็นระเบียบ
!unzip -q super-ai-engineer-ss-6-word-segmentation.zip -d dataset/

# แสดงรายการไฟล์ทั้งหมดในโฟลเดอร์ dataset เพื่อเช็คว่ามาครบไหม
!ls dataset/

100% 1.95M/1.95M [00:01<00:00, 1.11MB/s]

'LST20 Annotation Guideline.pdf'   ws_list.txt		      ws_test.txt
'LST20 Brief Specification.pdf'    ws_sample_submission.csv


In [ ]:
!pip install -q transformers==4.30.0 sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 31.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 145.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
# 1. สร้างโฟลเดอร์สำหรับเก็บข้อมูลที่แตกออกมา เพื่อความเป็นระเบียบ
!mkdir -p /content/lst20_data

# 2. สั่งแตกไฟล์จาก Path ที่คุณระบุ
# -q คือ quiet (ไม่ต้องแสดงชื่อไฟล์ทั้งหมด)
# -d คือ destination (ระบุโฟลเดอร์ปลายทาง)
!unzip -q /content/opend_lst20_corpus.zip -d /content/lst20_data

# 3. ตรวจสอบกระบวนการ: แสดงโครงสร้างไฟล์ที่ได้
print("รายการโฟลเดอร์ที่ได้จากการแตกไฟล์:")
!ls -F /content/lst20_data

replace /content/lst20_data/LST20_Corpus/AGREEMENT.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
รายการโฟลเดอร์ที่ได้จากการแตกไฟล์:
LST20_Corpus/


In [ ]:
# ดาวน์เกรดไลบรารี datasets เพื่อให้สามารถใช้งาน dataset script ของ LST20 ได้
!pip install -q datasets==2.19.0

In [ ]:
pip install datasets pandas

In [ ]:
!pip install -q sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.5 MB/s eta 0:00:00


In [ ]:
!pip install transformers datasets evaluate peft

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType

# 1. กำหนดชื่อโมเดลปราชญ์ภาษาไทย
model_name = "airesearch/wangchanberta-base-att-spm-uncased"

# 2. โหลด Tokenizer (ตัวสับคำของโมเดล)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. กำหนดป้ายกำกับ (Labels) ที่เราจะใช้
label2id = {'B_WORD': 0, 'I_WORD': 1, 'E_WORD': 2, 'whitespace': 3, 'O': 4}
id2label = {v: k for k, v in label2id.items()}

# 4. โหลดโมเดลสำหรับงานแปะป้ายกำกับ (Token Classification)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 5. การตั้งค่ากระดาษโพสต์อิท (LoRA Config)
# ทฤษฎี: เราจะแทรกเมทริกซ์ที่ชั้น query และ value ของ Attention Mechanism
lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, # บอกว่านี่คืองาน Token Classification
    inference_mode=False,
    r=8,              # ขนาด Rank ของเมทริกซ์ (ยิ่งน้อยยิ่ง Train ไว แต่ถ้าน้อยไปอาจโง่) 8 คือค่ามาตรฐานที่ดี
    lora_alpha=16,    # ตัวคูณขยายผลลัพธ์จาก LoRA
    lora_dropout=0.1, # ป้องกันโมเดลจำข้อสอบ
    target_modules=["query", "value"] # จุดที่เราจะแปะโพสต์อิทใน WangchanBERTa
)

# 6. ประกอบร่าง! เอาโมเดลหลักมาติด LoRA
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()
# คุณจะเห็นว่าพารามิเตอร์ที่ต้อง Train ลดลงเหลือหลักล้านต้นๆ จากร้อยล้าน!

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForTokenClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 298,757 || all params: 104,956,426 || trainable%: 0.2846


In [ ]:
from transformers import TrainingArguments

# 7. ตั้งค่าการฝึกสอน (Hyperparameters) ที่แก้ไขแล้ว
training_args = TrainingArguments(
    output_dir="./wangchanberta_lora_wordseg",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",  # <--- แก้ไขตรงนี้ครับ! จาก evaluation_strategy เป็น eval_strategy
    save_strategy="epoch",
    logging_steps=100,
)

In [ ]:
# สมมติว่า label2id ที่เรานิยามไว้คือ
# label2id = {'B_WORD': 0, 'I_WORD': 1, 'E_WORD': 2, 'whitespace': 3, 'O': 4}

def tokenize_and_align_labels(examples):
    """
    รับข้อมูลดิบ (ตัวอักษรและป้ายกำกับ) มาจัดเรียงให้ตรงกับ Tokenizer ของโมเดล
    """
    # 1. ให้ Tokenizer หั่นข้อมูล (is_split_into_words=True เพราะเราแยกตัวอักษรมาให้แล้ว)
    tokenized_inputs = tokenizer(
        examples["chars"],
        truncation=True,
        is_split_into_words=True,
        max_length=256
    )

    labels = []
    # 2. วนลูปเพื่อจับคู่ป้ายกำกับ (Label) ให้ตรงกับ Token ที่ถูกหั่น
    for i, label in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # ถ้าเป็น Special Token (เช่น <s>, </s>) ให้ใส่ค่า -100 (PyTorch จะข้ามการคิด Loss)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # ถ้าเป็น Token แรกของตัวอักษร ให้ใส่รหัส Label
                label_ids.append(label2id[label[word_idx]])
            else:
                # ถ้า Tokenizer หั่นตัวอักษรเดียวกันออกเป็นหลายชิ้น (Sub-word)
                # ชิ้นหลังๆ เราจะไม่นำมาคิด Loss ซ้ำ ให้ใส่ -100
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    # 3. นำ Label ที่จัดเรียงเสร็จแล้วยัดกลับเข้าไปใน Input
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForTokenClassification

# ---------------------------------------------------------
# ขั้นตอนที่ 1: จัดโครงสร้าง train_data ให้อยู่ในรูป Dictionary
# ---------------------------------------------------------
print("1. กำลังแปลงข้อมูลเข้าสู่รูปแบบ Dictionary...")
# train_data ปัจจุบันคือ: [ [('ส','B_WORD'), ('ม','I_WORD'),...], [...] ]
dict_data = {"chars": [], "tags": []}

for sentence in train_data:
    chars = [item[0] for item in sentence] # ดึงมาเฉพาะตัวอักษร
    tags = [item[1] for item in sentence]  # ดึงมาเฉพาะป้ายกำกับ
    dict_data["chars"].append(chars)
    dict_data["tags"].append(tags)

# ---------------------------------------------------------
# ขั้นตอนที่ 2: สร้าง Hugging Face Dataset และแบ่งส่วน
# ---------------------------------------------------------
print("2. สร้างสายพานลำเลียงข้อมูล (HF Dataset)...")
hf_dataset = Dataset.from_dict(dict_data)

# แบ่ง 10% ไว้เป็น Validation Set เพื่อเช็กว่าโมเดลเรียนรู้ได้ดีแค่ไหน
dataset_split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split['train']
valid_dataset = dataset_split['test']

# ---------------------------------------------------------
# ขั้นตอนที่ 3: จับข้อมูลมา Tokenize และ Align Labels
# ---------------------------------------------------------
print("3. ทำการจับคู่ตัวอักษรกับ Token (อาจใช้เวลา 1-2 นาที)...")
# ใช้ .map() เพื่อรันฟังก์ชัน tokenize_and_align_labels (ที่คุณเพิ่งแก้ไข eval_strategy ไป)
train_tokenized = train_dataset.map(tokenize_and_align_labels, batched=True)
valid_tokenized = valid_dataset.map(tokenize_and_align_labels, batched=True)



1. กำลังแปลงข้อมูลเข้าสู่รูปแบบ Dictionary...
2. สร้างสายพานลำเลียงข้อมูล (HF Dataset)...
3. ทำการจับคู่ตัวอักษรกับ Token (อาจใช้เวลา 1-2 นาที)...


Map:   0%|          | 0/56979 [00:00<?, ? examples/s]

Map:   0%|          | 0/6331 [00:00<?, ? examples/s]

4. เตรียมเครื่องจักร Trainer...


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
# ---------------------------------------------------------
# ขั้นตอนที่ 4: สร้าง Data Collator และ Trainer (ฉบับแก้ไข)
# ---------------------------------------------------------
print("4. เตรียมเครื่องจักร Trainer...")
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    data_collator=data_collator    # <--- เหลือแค่นี้พอ ลบ tokenizer ออกไปเลย!
)

# ---------------------------------------------------------
# ขั้นตอนที่ 5: เริ่มฝึกสอน และเซฟ Tokenizer!
# ---------------------------------------------------------
print("5. 🚀 เริ่มกระบวนการ Fine-tune ด้วย LoRA ทะยานสู่ Leaderboard!")
trainer.train()

# เนื่องจากเราเอา tokenizer ออกจาก Trainer ตอนเซฟโมเดลเราจึงต้องสั่งเซฟมันแยกด้วยครับ
print("6. กำลังบันทึกโมเดลและ Tokenizer...")
trainer.save_model("./wangchanberta_lora_wordseg_final")
tokenizer.save_pretrained("./wangchanberta_lora_wordseg_final")
print("🎉 ทุกอย่างเสร็จสมบูรณ์!")

4. เตรียมเครื่องจักร Trainer...
5. 🚀 เริ่มกระบวนการ Fine-tune ด้วย LoRA ทะยานสู่ Leaderboard!


`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,0.182357,0.154152
2,0.157836,0.129315
3,0.149657,0.121852


6. กำลังบันทึกโมเดลและ Tokenizer...
🎉 ทุกอย่างเสร็จสมบูรณ์!


In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import PeftModel

# ---------------------------------------------------------
# ขั้นตอนที่ 1: โหลดโมเดลที่เราเพิ่ง Train เสร็จ
# ---------------------------------------------------------
print("1. กำลังโหลดปราชญ์ WangchanBERTa + LoRA ของเรา...")
base_model_name = "airesearch/wangchanberta-base-att-spm-uncased"
model_dir = "./wangchanberta_lora_wordseg_final"

label2id = {'B_WORD': 0, 'I_WORD': 1, 'E_WORD': 2, 'whitespace': 3, 'O': 4}
id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(model_dir)
base_model = AutoModelForTokenClassification.from_pretrained(
    base_model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)
model = PeftModel.from_pretrained(base_model, model_dir)

# ย้ายโมเดลไปอยู่บน GPU (ถ้ามี) เพื่อความรวดเร็ว
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # สั่งให้โมเดลเข้าโหมดทำนายผล (ห้ามเรียนรู้เพิ่ม)

# ---------------------------------------------------------
# ขั้นตอนที่ 2: อ่านข้อสอบและหั่นเป็นท่อนๆ
# ---------------------------------------------------------
print("2. กำลังอ่านและหั่นข้อสอบ ws_test.txt...")
test_file_path = "/content/dataset/ws_test.txt"
with open(test_file_path, "r", encoding="utf-8") as f:
    test_text = f.read()

test_chars = list(test_text)
chunk_size = 200 # หั่นทีละ 200 ตัวอักษร
all_predictions = []



1. กำลังโหลดปราชญ์ WangchanBERTa + LoRA ของเรา...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForTokenClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2. กำลังอ่านและหั่นข้อสอบ ws_test.txt...


In [ ]:
# ---------------------------------------------------------
# ขั้นตอนที่ 3: ลงมือทำนายผลทีละท่อน (Inference Loop - แก้ไขแล้ว)
# ---------------------------------------------------------
print("3. เริ่มทำนายผล (อาจใช้เวลาสักครู่)...")
for i in range(0, len(test_chars), chunk_size):
    chunk_chars = test_chars[i : i+chunk_size]

    # 1. ให้ Tokenizer จัดการแปลง (ได้ BatchEncoding object)
    inputs_encoded = tokenizer(chunk_chars, is_split_into_words=True, return_tensors="pt")

    # 2. ดึง word_ids ออกมาก่อนเลย! (แก้บั๊กตรงนี้)
    word_ids = inputs_encoded.word_ids()

    # 3. ย้ายข้อมูลเข้า GPU โดยแปลงเป็น dict ปกติ
    inputs = {k: v.to(device) for k, v in inputs_encoded.items()}

    with torch.no_grad(): # ปิดการคำนวณ Gradient เพื่อประหยัดแรม
        outputs = model(**inputs)

    logits = outputs.logits
    # หาค่า Label ที่มีความน่าจะเป็นสูงสุด
    preds = torch.argmax(logits, dim=2).squeeze().tolist()

    # ถ้า predict ท่อนสั้นๆ preds อาจจะไม่เป็น list เราต้องดักไว้
    if isinstance(preds, int):
        preds = [preds]

    # 4. จัดเรียงคำตอบกลับไปเป็นระดับตัวอักษร (Alignment)
    previous_word_idx = None
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue # ข้าม special tokens
        if word_idx != previous_word_idx:
            # เก็บเฉพาะโทเคนแรกของตัวอักษรนั้นๆ
            all_predictions.append(id2label[preds[idx]])
        previous_word_idx = word_idx

print(f"ทำนายเสร็จสิ้น! จำนวนคำตอบที่ได้: {len(all_predictions):,} ตัว (ต้องตรงกับ 37,248)")

3. เริ่มทำนายผล (อาจใช้เวลาสักครู่)...
ทำนายเสร็จสิ้น! จำนวนคำตอบที่ได้: 35,182 ตัว (ต้องตรงกับ 37,248)


In [ ]:
import torch
import pandas as pd

# ==========================================
# ส่วนที่ 1: ทำนายผลและเตรียมคำตอบ 37,248 ตัว
# ==========================================
print("1. กำลังโหลดข้อสอบและทำนายผล...")
test_file_path = "/content/dataset/ws_test.txt"
with open(test_file_path, "r", encoding="utf-8") as f:
    test_chars = list(f.read())

chunk_size = 200
all_predictions = [] # รีเซ็ตตัวแปรใหม่ให้สะอาด!

for i in range(0, len(test_chars), chunk_size):
    chunk_chars = test_chars[i : i+chunk_size]

    # 1.1 ให้ Tokenizer จัดการแปลง
    inputs_encoded = tokenizer(chunk_chars, is_split_into_words=True, return_tensors="pt")
    word_ids = inputs_encoded.word_ids()

    # 1.2 ย้ายเข้า GPU
    inputs = {k: v.to(device) for k, v in inputs_encoded.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    preds = torch.argmax(logits, dim=2).squeeze().tolist()
    if isinstance(preds, int): preds = [preds]

    # 1.3 สร้างกล่อง dummy (whitespace) รอไว้
    chunk_predictions = ['whitespace'] * len(chunk_chars)

    previous_word_idx = None
    for idx, word_idx in enumerate(word_ids):
        if word_idx is None: continue
        if word_idx != previous_word_idx:
            # ป้องกัน Index out of bound เผื่อ Tokenizer รวน
            if word_idx < len(chunk_predictions):
                chunk_predictions[word_idx] = id2label[preds[idx]]
        previous_word_idx = word_idx

    all_predictions.extend(chunk_predictions)

print(f"-> ทำนายเสร็จสิ้น! ได้คำตอบมาทั้งหมด: {len(all_predictions):,} ตัว (เป้าหมาย: 37,248)")

# ==========================================
# ส่วนที่ 2: หยอดลง Template 35,182 แถว
# ==========================================
print("\n2. กำลังสร้างไฟล์ Submission...")
sample_sub_path = "/content/dataset/ws_sample_submission.csv"
sample_df = pd.read_csv(sample_sub_path)
required_ids = sample_df['Id'].values

submission_data = []

for idx in required_ids:
    # กันเหนียวชั้นที่ 2: เช็กว่า Id ไม่เกินความยาวของ Array
    if idx - 1 < len(all_predictions):
        pred_label = all_predictions[idx - 1]
    else:
        pred_label = 'I_WORD' # ถ้าหาไม่เจอ ให้เป็น I_WORD แทน

    if pred_label not in ['B_WORD', 'I_WORD', 'E_WORD']:
        pred_label = 'I_WORD'

    submission_data.append({'Id': idx, 'Predicted': pred_label})

df_submission = pd.DataFrame(submission_data)
output_filename = "wangchanberta_lora_submission_final.csv"
df_submission.to_csv(output_filename, index=False)

print(f"-> 🎉 เสร็จสมบูรณ์! บันทึก '{output_filename}' เรียบร้อย")
print(f"-> จำนวนแถวที่ได้: {len(df_submission):,} แถว (เป้าหมาย: 35,182)")

1. กำลังโหลดข้อสอบและทำนายผล...
-> ทำนายเสร็จสิ้น! ได้คำตอบมาทั้งหมด: 37,248 ตัว (เป้าหมาย: 37,248)

2. กำลังสร้างไฟล์ Submission...
-> 🎉 เสร็จสมบูรณ์! บันทึก 'wangchanberta_lora_submission_final.csv' เรียบร้อย
-> จำนวนแถวที่ได้: 35,182 แถว (เป้าหมาย: 35,182)


In [ ]:
from transformers import TrainingArguments, Trainer

print("กำลังเริ่มฝึกสอนโมเดล V2 (เรียนรู้ลึกขึ้น)...")
# 1. สร้าง Training Arguments ตัวใหม่
training_args_v2 = TrainingArguments(
    output_dir="./wangchanberta_lora_v2",
    learning_rate=1e-4,             # ลดลงจาก 2e-4 เพื่อให้ปรับค่าน้ำหนักละเอียดขึ้น
    num_train_epochs=5,             # เพิ่มรอบจาก 3 เป็น 5 (เรามีเวลาพอ)
    per_device_train_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
)

# 2. สั่ง Trainer เริ่มทำงาน (ใช้ data_collator และ dataset เดิมได้เลย)
trainer_v2 = Trainer(
    model=lora_model,
    args=training_args_v2,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    data_collator=data_collator
)

# ขีดคอมเมนต์ออกเพื่อเริ่ม Train!


กำลังเริ่มฝึกสอนโมเดล V2 (เรียนรู้ลึกขึ้น)...


In [ ]:
 trainer_v2.train()

Epoch,Training Loss,Validation Loss
1,0.134245,0.111845
2,0.129252,0.106568
3,0.126751,0.098765
4,0.119693,0.099677
5,0.119753,0.097383


TrainOutput(global_step=17810, training_loss=0.12904960342098795, metrics={'train_runtime': 9774.9665, 'train_samples_per_second': 29.145, 'train_steps_per_second': 1.822, 'total_flos': 3.735281497618944e+16, 'train_loss': 0.12904960342098795, 'epoch': 5.0})

In [ ]:
import pandas as pd

# สมมติว่า all_predictions คือ list คำตอบ 37,248 ตัวของคุณ
# และ test_chars คือ list ตัวอักษรข้อสอบ 37,248 ตัว

def apply_post_processing_rules(chars, preds):
    print("กำลังตรวจสอบและแก้ไขข้อผิดพลาดด้วยกฎ (Post-processing)...")
    refined_preds = preds.copy()
    fix_count = 0

    # เริ่มเช็กตั้งแต่ตัวที่ 2 เป็นต้นไป
    for i in range(1, len(chars)):
        char_current = chars[i]
        char_prev = chars[i-1]

        # กฎข้อที่ 1: กลุ่มตัวเลข (0-9) ต้องไม่ถูกตัดแยกออกจากกัน
        if char_current.isdigit() and char_prev.isdigit():
            if refined_preds[i] != 'I_WORD':
                refined_preds[i] = 'I_WORD'
                fix_count += 1

        # กฎข้อที่ 2: กลุ่มตัวอักษรภาษาอังกฤษ ต้องไม่ถูกตัดแยกออกจากกัน
        elif char_current.isascii() and char_current.isalpha() and char_prev.isascii() and char_prev.isalpha():
            if refined_preds[i] != 'I_WORD':
                refined_preds[i] = 'I_WORD'
                fix_count += 1

    print(f"แก้ไขจุดที่โมเดลพลาดไปทั้งหมด: {fix_count} จุด!")
    return refined_preds

# 1. ใช้งานฟังก์ชัน
refined_predictions = apply_post_processing_rules(test_chars, all_predictions)

# 2. หยอดคำตอบลงไฟล์ CSV 35,182 แถว
print("กำลังสร้างไฟล์ Submission (V1 + Post-processing)...")
sample_df = pd.read_csv("ws_sample_submission.csv")
required_ids = sample_df['Id'].values
submission_data = []

for idx in required_ids:
    if idx - 1 < len(refined_predictions):
        pred_label = refined_predictions[idx - 1]
    else:
        pred_label = 'I_WORD'

    if pred_label not in ['B_WORD', 'I_WORD', 'E_WORD']:
        pred_label = 'I_WORD'

    submission_data.append({'Id': idx, 'Predicted': pred_label})

# 3. บันทึกเป็นไฟล์ใหม่
df_refined_submission = pd.DataFrame(submission_data)
output_filename = "wangchanberta_lora_refined_submission.csv"
df_refined_submission.to_csv(output_filename, index=False)
print(f"บันทึกไฟล์ '{output_filename}' เรียบร้อย พร้อมส่ง!")